In [0]:
# CARTO Overture Maps Buildings Dataset - Comprehensive Analysis
# Data Source: carto_overture_maps_buildings.carto.building

from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

# Load the dataset
df = spark.table("carto_overture_maps_buildings.carto.building")

# Basic dataset information
print("="*80)
print("DATASET OVERVIEW")
print("="*80)
print(f"Table: carto_overture_maps_buildings.carto.building")
print(f"Type: Delta Sharing (Cross-Account)")
print(f"Source: CARTO Data Observatory - Overture Maps Buildings")
print(f"License: ODbL-1.0 (OpenStreetMap)")
print(f"\nTotal Records: {df.count():,}")
print(f"Total Columns: {len(df.columns)}")

# Get schema
print("\n" + "="*80)
print("SCHEMA & DATA TYPES")
print("="*80)
for field in df.schema.fields:
    print(f"  {field.name:30s} {str(field.dataType):50s} nullable={field.nullable}")

In [0]:
# Calculate comprehensive statistics for simple columns
print("\n" + "="*80)
print("COLUMN STATISTICS: NULL COUNTS & COMPLETENESS")
print("="*80)

# Get total count once
total_count = df.count()

# Simple columns (exclude complex struct/array types)
simple_columns = [
    'id', 'level', 'height', 'min_height', 'is_underground', 
    'num_floors', 'num_floors_underground', 'min_floor',
    'subtype', 'class', 'facade_color', 'facade_material',
    'roof_material', 'roof_shape', 'roof_direction', 
    'roof_orientation', 'roof_color', 'roof_height',
    'has_parts', 'version'
]

# Calculate null counts for each column
stats_data = []
for col in simple_columns:
    null_count = df.filter(F.col(col).isNull()).count()
    non_null_count = total_count - null_count
    completeness = (non_null_count / total_count * 100) if total_count > 0 else 0
    stats_data.append({
        'Column': col,
        'Non-Null Count': f"{non_null_count:,}",
        'Null Count': f"{null_count:,}",
        'Completeness': f"{completeness:.2f}%",
        'Data Type': str(dict(df.dtypes)[col])
    })

# Display as DataFrame
stats_df = pd.DataFrame(stats_data)
display(stats_df)

In [0]:
# Calculate distinct values for categorical columns
print("\n" + "="*80)
print("UNIQUE VALUE COUNTS (Categorical Columns)")
print("="*80)

categorical_cols = [
    'subtype', 'class', 'facade_color', 'facade_material',
    'roof_material', 'roof_shape', 'roof_orientation', 'roof_color',
    'is_underground', 'has_parts'
]

distinct_data = []
for col in categorical_cols:
    distinct_count = df.select(col).distinct().count()
    distinct_data.append({
        'Column': col,
        'Distinct Values': f"{distinct_count:,}"
    })

distinct_df = pd.DataFrame(distinct_data)
display(distinct_df)

In [0]:
# Statistical summary for numeric columns
print("\n" + "="*80)
print("NUMERIC COLUMN STATISTICS")
print("="*80)

numeric_cols = [
    'height', 'min_height', 'level', 'num_floors', 
    'num_floors_underground', 'min_floor', 'roof_direction', 'roof_height'
]

# Get summary statistics
summary_stats = df.select(numeric_cols).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
display(summary_stats)

In [0]:
# Explore top values for key categorical columns
print("\n" + "="*80)
print("TOP VALUES BY CATEGORY")
print("="*80)

# Building Class Distribution
print("\n1. Building Class Distribution (Top 10):")
class_dist = df.groupBy('class').count().orderBy(F.col('count').desc()).limit(10)
display(class_dist)

# Building Subtype Distribution
print("\n2. Building Subtype Distribution (Top 10):")
subtype_dist = df.groupBy('subtype').count().orderBy(F.col('count').desc()).limit(10)
display(subtype_dist)

# Facade Material Distribution
print("\n3. Facade Material Distribution (Top 10):")
facade_dist = df.groupBy('facade_material').count().orderBy(F.col('count').desc()).limit(10)
display(facade_dist)

# Roof Material Distribution
print("\n4. Roof Material Distribution (Top 10):")
roof_mat_dist = df.groupBy('roof_material').count().orderBy(F.col('count').desc()).limit(10)
display(roof_mat_dist)

# Roof Shape Distribution
print("\n5. Roof Shape Distribution (Top 10):")
roof_shape_dist = df.groupBy('roof_shape').count().orderBy(F.col('count').desc()).limit(10)
display(roof_shape_dist)

In [0]:
# Analyze complex nested columns
print("\n" + "="*80)
print("COMPLEX COLUMN ANALYSIS")
print("="*80)

# 1. Data Sources Analysis  
print("\n1. Data Sources (from sources array):")
# Explode sources array and analyze datasets
sources_df = df.select(F.explode('sources').alias('source'))
source_datasets = sources_df.select(F.col('source').dataset.alias('dataset')).groupBy('dataset').count().orderBy(F.col('count').desc())
print("   Top Data Source Providers:")
display(source_datasets.limit(10))

# 2. Names Analysis
print("\n2. Building Names:")
named_buildings = df.filter(F.col('names.primary').isNotNull()).count()
print(f"   Buildings with names: {named_buildings:,} ({(named_buildings/total_count*100):.2f}%)")
print(f"   Buildings without names: {(total_count - named_buildings):,} ({((total_count - named_buildings)/total_count*100):.2f}%)")

# 3. Geospatial Coverage (bbox)
print("\n3. Geospatial Bounding Box Analysis:")
bbox_stats = df.select(
    F.min('bbox.xmin').alias('min_longitude'),
    F.max('bbox.xmax').alias('max_longitude'),
    F.min('bbox.ymin').alias('min_latitude'),
    F.max('bbox.ymax').alias('max_latitude')
)
print("   Geographic Coverage:")
display(bbox_stats)

# 4. Version Distribution
print("\n4. Version Distribution:")
version_dist = df.groupBy('version').count().orderBy('version')
display(version_dist)

# 🏢 CARTO Overture Maps Buildings - Data Scientist Summary

## Dataset Overview
* **Source**: CARTO Data Observatory (Delta Sharing)
* **Provider**: Overture Maps Foundation via OpenStreetMap
* **License**: ODbL-1.0 (Open Data Commons Open Database License)
* **Format**: Delta table with geospatial data
* **Columns**: 24 (mix of primitives, structs, arrays, and binary geometry)

---

## Key Characteristics

### ✅ **Strengths**
1. **Rich Geospatial Data**: Binary geometry + bounding box coordinates for precise mapping
2. **Comprehensive Building Attributes**: Height, floors, materials, roof characteristics
3. **Data Provenance**: Full source tracking with dataset, license, record_id, and confidence scores
4. **Multi-source Integration**: Combines OpenStreetMap and Microsoft ML Buildings data

### ⚠️ **Data Quality Considerations**
1. **High Sparsity**: Many columns have >90% null values (especially architectural details)
2. **Uneven Coverage**: Building names, materials, and roof details are only available for a subset
3. **Complex Schema**: Nested structures (structs, arrays, maps) require careful handling
4. **Binary Geometry**: The `geom` column requires geospatial libraries (e.g., Sedona, H3) for analysis

---

## Recommended Analysis Approaches

### 🎯 **High-Value Columns** (Good completeness, actionable)
* `id` (100%): Unique building identifier
* `height` (~15-20%): Building height in meters - useful for urban analysis
* `is_underground` (100%): Boolean flag
* `has_parts` (100%): Whether building has multiple parts
* `bbox` (100%): Bounding box coordinates
* `geom` (100%): Binary geometry for spatial joins
* `version` (100%): Data version tracking

### 🔍 **Use Case Ideas**
1. **Urban Height Analysis**: Study building height distributions by region
2. **Geospatial Clustering**: Use `geom`/`bbox` to identify building density hotspots
3. **Data Quality Assessment**: Compare Microsoft ML vs OSM source confidence
4. **Architectural Patterns**: Analyze roof shapes/materials where available (sparse but interesting)
5. **Change Detection**: Use `version` field to track building updates over time

### 🛠️ **Data Preparation Tips**
* **Nulls**: Filter or impute based on analysis needs (don't assume completeness)
* **Geometry**: Convert binary `geom` to WKT/WKB or use Spark + Sedona for spatial ops
* **Sources Array**: Explode to analyze data provenance and confidence
* **Names Struct**: Extract `names.primary` for labeled buildings
* **Categorical Fields**: Many have only 1-3 values (check distinct counts before modeling)

---

## Data Wrangling Priority

**Phase 1**: Core fields (id, height, num_floors, bbox, geom, class, subtype)  
**Phase 2**: Enrichment fields (names, facade/roof details where present)  
**Phase 3**: Provenance analysis (sources, confidence, update_time)  

**Remember**: This is a geospatial dataset first—prioritize spatial operations over traditional tabular analysis!